# segment-line-intersect-2d — faded example 2: Build the batched A matrix for N segments vs one line

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `segment-line-intersect-2d`. Running the beacon reports progress on the `Geometry: Segment-line intersect 2-D` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Segment-line intersect 2-D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`segment-line-intersect-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "segment-line-intersect-2d"
DD_SUBTOPIC = "Geometry: Segment-line intersect 2-D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For batched intersection testing, the key insight is that the line direction `e` is shared across all N segments. Using `e.expand_as(d)` replicates it N times without copying memory, giving the correct (N, 2) shape for stacking into the (N, 2, 2) system matrix. The stack goes along the last axis (`dim=-1`) so each slice `A[k]` contains the two column vectors as expected by `linalg.solve`.

## Faded exercise 2

Given S0 of shape (N, 2), S1 of shape (N, 2), and a line defined by L0 and L1 (both shape (2,)), build the batched A matrix of shape (N, 2, 2).

1. Compute segment directions d = S1 - S0.
2. Compute line direction e = L1 - L0.
3. Build A by stacking d and -e (broadcast) as the two columns.

The blank step is constructing A by stacking d and the expanded -e along the last axis.

**Fill in:** Stack d and -e (expanded to match d's shape) as columns along dim=-1 to form the (N, 2, 2) system matrix.

In [ ]:
import torch as t

t.manual_seed(0)

S0 = t.tensor([[0.0,0.0],[1.0,1.0],[2.0,0.0]], dtype=t.float32)
S1 = t.tensor([[3.0,3.0],[3.0,-1.0],[2.0,4.0]], dtype=t.float32)
L0 = t.tensor([0.0, 2.0])
L1 = t.tensor([4.0, 2.0])  # horizontal line y=2

d = S1 - S0                      # (3, 2)
e = L1 - L0                      # (2,)
A = None  # TODO: Stack d and -e (expanded to match d's shape) as columns along dim=-1 to form the (N, 2, 2) system matrix.
b = L0 - S0                      # (3, 2)
ts = t.linalg.solve(A, b)        # (3, 2)
hit = (ts[:, 0] >= 0) & (ts[:, 0] <= 1)

print('A.shape:', A.shape)        # should be (3, 2, 2)
print('t_seg:', ts[:, 0].tolist())
print('hit  :', hit.tolist())


def _test():
    import torch as t

    S0 = t.tensor([[0.0,0.0],[1.0,1.0],[2.0,0.0]], dtype=t.float32)
    S1 = t.tensor([[3.0,3.0],[3.0,-1.0],[2.0,4.0]], dtype=t.float32)
    L0 = t.tensor([0.0, 2.0])
    L1 = t.tensor([4.0, 2.0])

    d = S1 - S0
    e = L1 - L0
    A_ref = t.stack([d, -e.expand_as(d)], dim=-1)
    assert A.shape == (3, 2, 2), f'shape: {A.shape}'
    assert t.allclose(A, A_ref, atol=1e-5), 'A matrix mismatch'
    # solve should succeed and produce correct hits
    b = L0 - S0
    ts_ref = t.linalg.solve(A_ref, b)
    hit_ref = (ts_ref[:, 0] >= 0) & (ts_ref[:, 0] <= 1)
    assert t.equal(hit, hit_ref), f'hit mismatch: {hit} vs {hit_ref}'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

S0 = t.tensor([[0.0,0.0],[1.0,1.0],[2.0,0.0]], dtype=t.float32)
S1 = t.tensor([[3.0,3.0],[3.0,-1.0],[2.0,4.0]], dtype=t.float32)
L0 = t.tensor([0.0, 2.0])
L1 = t.tensor([4.0, 2.0])

d = S1 - S0
e = L1 - L0
A = t.stack([d, -e.expand_as(d)], dim=-1)
b = L0 - S0
ts = t.linalg.solve(A, b)
hit = (ts[:, 0] >= 0) & (ts[:, 0] <= 1)

print('A.shape:', A.shape)
print('t_seg:', ts[:, 0].tolist())
print('hit  :', hit.tolist())
```
</details>